# Appliance Energy Forecasting — narrative walkthrough

This notebook runs the same pipeline as `run_pipeline.py`, part by part, so you can inspect intermediate results and figures. Run top to bottom. Use `synthetic=True` in the load cell for a quick offline pass.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from src import config, data, eda, benchmarks, sarimax_model, ml_model, foundation, evaluate
from src.utils import plot_forecast

## Part 1 — load, resample to hourly, EDA & stationarity

In [ ]:
df = data.load(synthetic=False)   # set synthetic=True to skip the download
print(df.shape)
display(df.head())
print(data.missing_report(df))

In [ ]:
stat = eda.run(df)
stat   # ADF + KPSS on level, 1st diff and seasonal diff

Figures written to `outputs/figures/` (01–05): full series, first week, seasonal profiles, decomposition, ACF/PACF.

## Part 2–3 — problem definition & benchmarks
Target = `Appliances`, 24-hour horizon, final 24h held out.

In [ ]:
y = df[config.TARGET]
train, test = y.iloc[:-config.HORIZON], y.iloc[-config.HORIZON:]
bench = benchmarks.run_all(train, test)
forecasts = {n:f for n,(f,m) in bench.items()}
metrics = {n:m for n,(f,m) in bench.items()}
best_bench = min(bench, key=lambda k: bench[k][1]['RMSE'])
print('strongest benchmark:', best_bench)
import pandas as pd; pd.DataFrame(metrics).T.sort_values('RMSE')

## Part 4 — SARIMAX (AIC grid search)
Use `fast=True` for a quick grid; the full grid loops p,d,q ∈ [0,6],[0,2],[0,6].

In [ ]:
sx = sarimax_model.run(df, fast=True, verbose=False)
metrics['SARIMAX'] = sx['metrics']; forecasts['SARIMAX'] = sx['forecast']
print('order', sx['order'], sx['seasonal_order'], 'AIC', round(sx['aic'],1))
print('Ljung-Box p =', round(sx['diagnostics']['ljung_box_pvalue'],3))
plot_forecast(sx['train_tail'], sx['test'], sx['forecast'], 'SARIMAX 24h', '10_sarimax_forecast.png', lower=sx['lower'], upper=sx['upper'])

## Part 5–6 — features & XGBoost

In [ ]:
xgb = ml_model.run(df)
metrics['XGBoost'] = xgb['metrics']; forecasts['XGBoost'] = xgb['forecast']
print('24h RMSE', round(xgb['metrics']['RMSE'],2), '| 14d RMSE', round(xgb['metrics_14d']['RMSE'],2))
print(xgb['group_importance'])
plot_forecast(xgb['train_tail'], xgb['test'], xgb['forecast'], 'XGBoost 24h', '11_xgb_forecast.png')

## Part 7 — Chronos foundation model
Downloads a small model on first use; falls back to seasonal-naive if unavailable.

In [ ]:
fm = foundation.run(df)
metrics[fm['name']] = fm['metrics']; forecasts[fm['name']] = fm['forecast']
print(fm['name'], 'RMSE', round(fm['metrics']['RMSE'],2))

## Part 8 — evaluation

In [ ]:
comparison = evaluate.comparison_table(metrics)
evaluate.all_forecasts_plot(test, forecasts, y)
evaluate.error_diagnostics(test, forecasts, best_bench)
evaluate.skill_scores(comparison, best_bench).round(2)